# Phase 2: 独自くるみDataset

Phase 1と同じPipeline（`run_experiment`）で、スマートフォンで撮影したくるみ画像を評価します。Dataset固有の処理はAdapter（`dataset.type: walnut`）だけです。

- 正常: 明確な欠け・割れがない（色・大きさ・形状・模様の個体差は正常）
- 異常: `crack`（割れ）、`chip`（欠け）
- 学習: 正常画像のみ（split=train） / Threshold決定: validation / 最終評価: test
- Ground Truth Maskは作成しないため、Image-level評価のみ行います

## 1. Environment

In [ ]:
%matplotlib inline
from pathlib import Path

import pandas as pd
from IPython.display import display

from anomaly_detection.utils.config import find_project_root, load_config
from anomaly_detection.utils.device import describe_device, resolve_device
from anomaly_detection.utils.logging import setup_logging
from anomaly_detection.utils.reproducibility import collect_environment

setup_logging()
PROJECT_ROOT = find_project_root(Path.cwd())
env = collect_environment()
print("Python", env["python"], "|", env["platform"])
display(pd.Series(env["packages"], name="version").to_frame())

In [ ]:
# 利用可能なDevice（実行時は設定の runtime.device に従う）
describe_device(resolve_device("auto"))

## 2. Config

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "phase2_walnut.yaml"
config = load_config(CONFIG_PATH)
print("Config:", config.config_path)
config.to_dict()

## 3. Manifest作成（必要な場合のみ）

`raw/good`, `raw/crack`, `raw/chip` に画像を配置した後、`BUILD_MANIFEST = True` にして実行すると`manifest.csv` を作成します。split比率は設定の `dataset.split` で変更でき、同じ `object_id`（ファイル名の `_` より前）の画像は同じsplitに入ります。

既存のManifestは上書きしません（作り直す場合は `OVERWRITE = True`）。

In [ ]:
from anomaly_detection.datasets import create_adapter
from anomaly_detection.datasets.base import DatasetError

BUILD_MANIFEST = False
OVERWRITE = False

adapter = create_adapter(config)
if BUILD_MANIFEST:
    try:
        adapter.build_manifest(overwrite=OVERWRITE)
    except DatasetError as e:
        print(e)

## 4. Dataset Validation

Manifestの必須列・ファイル存在・label / defect_type / split の値・Trainへの異常画像混入・object_idのsplit跨ぎ（Data Leakage）を検証します。Datasetが未配置の場合は、必要なDirectory構成とManifest作成方法を表示して以降をスキップします。

In [ ]:
from anomaly_detection.datasets.base import DatasetNotFoundError
from anomaly_detection.datasets.walnut import (
    count_by_defect_type,
    find_object_leaks,
    load_manifest,
)

DATA_READY = False
try:
    samples = adapter.load_samples()
    DATA_READY = True
except DatasetNotFoundError as e:
    print(e)
except DatasetError as e:
    print("Datasetに不備があります。Manifestを確認してください。")
    print(e)

In [ ]:
if DATA_READY:
    print("件数（split × good / crack / chip）")
    display(count_by_defect_type(samples))
    print("個体数（object_id）")
    display(samples.groupby("split")["object_id"].nunique().to_frame("objects"))
    leaks = find_object_leaks(load_manifest(adapter.manifest_path))
    print("object_idのsplit跨ぎ:", "なし" if leaks.empty else f"{len(leaks)} 件")

## 5. Dataset Overview

In [ ]:
from matplotlib.figure import Figure

from anomaly_detection.datasets.base import read_image


def show_samples(rows: pd.DataFrame, title: str) -> Figure:
    figure = Figure(figsize=(3 * len(rows), 3.6))
    axes = figure.subplots(1, len(rows), squeeze=False)[0]
    for ax, (_, row) in zip(axes, rows.iterrows(), strict=True):
        ax.imshow(read_image(row["image_path"]))
        ax.set_title(Path(row["image_path"]).name, fontsize=8)
        ax.axis("off")
    figure.suptitle(title)
    return figure


if DATA_READY:
    for kind in ("good", "crack", "chip"):
        rows = samples[samples["defect_type"].replace("", "good") == kind]
        if not rows.empty:
            display(show_samples(rows.sample(min(4, len(rows)), random_state=0), kind))

## 6. Preprocessing

前処理のON/OFFとパラメータは `configs/phase2_walnut.yaml` の `preprocessing` で変更します（背景を除く場合は `crop`、照明ムラには `clahe` 等）。

In [ ]:
from anomaly_detection.preprocessing.pipeline import ImagePreprocessor, compare_images

preprocessor = ImagePreprocessor(config.preprocessing)
print("Preprocessing:", preprocessor.summary())
if DATA_READY:
    original = read_image(samples.iloc[0]["image_path"])
    display(compare_images(original, preprocessor.process(original)))

## 7. Training

Phase 1と同じ `run_experiment` で、学習 → Model保存 → 推論 → Threshold決定（validation） → 評価（test） → 可視化 → Run保存 → `runs/summary.csv` 追記までを実行します。

In [ ]:
from anomaly_detection.experiments.pipeline import run_experiment

result = run_experiment(config) if DATA_READY else None
if result:
    print("Run:", result.run.run_dir)

## 8. Inference

In [ ]:
if result:
    predictions = result.predictions
    display(predictions.head(10))
    display(
        predictions.groupby(["split", "ground_truth", "defect_type"])["anomaly_score"].describe()
    )

## 9. Metrics

最終評価は **test** のみです。validationの値はThreshold決定に使ったデータでの参考値です。

In [ ]:
from anomaly_detection.evaluation.metrics import confusion_matrix_frame

if result:
    metrics = result.metrics
    print("Threshold:", metrics["threshold"])
    keys = [
        "recall",
        "precision",
        "f1",
        "accuracy",
        "auroc",
        "aupr",
        "num_images",
        "num_good",
        "num_defect",
    ]
    columns = {"test (final)": {k: metrics["image"][k] for k in keys}}
    if "threshold_selection_reference" in metrics:
        reference = metrics["threshold_selection_reference"]["image"]
        columns["validation (reference)"] = {k: reference[k] for k in keys}
    display(pd.DataFrame(columns))
    display(confusion_matrix_frame(metrics["image"]))
    test = result.predictions[result.predictions["split"] == "test"]
    defects = test[test["ground_truth"] == "defect"]
    print("欠陥種別ごとの検出数（test）")
    display(pd.crosstab(defects["defect_type"], defects["prediction"]))

## 10. Visualization

In [ ]:
from anomaly_detection.visualization.anomaly_visualizer import load_display_image, render_result


def show_result(position: int) -> Figure:
    row = result.predictions.iloc[position]
    anomaly_map = result.anomaly_maps[position]
    image = load_display_image(row["image_path"], preprocessor, anomaly_map.shape)
    return render_result(image, anomaly_map, row, result.heatmap_scale)


if result:
    test_rows = result.predictions[result.predictions["split"] == "test"]
    for kind in ("crack", "chip", ""):
        matched = test_rows[test_rows["defect_type"] == kind]
        if not matched.empty:
            display(show_result(matched.index[0]))

## 11. Error Analysis

testの誤判定を、見逃し（FN）→ 過検知（FP）の順に表示します。

In [ ]:
from anomaly_detection.evaluation.error_analysis import get_errors, plot_errors

if result:
    errors = get_errors(result.predictions, split="test", fn_first=True)
    display(errors["error_type"].value_counts())
    columns = ["error_type", "image_path", "defect_type", "prediction", "anomaly_score"]
    display(errors[columns])
    figures = plot_errors(
        errors, result.anomaly_maps, preprocessor, result.heatmap_scale, max_items=6
    )
    for figure in figures:
        display(figure)

### Thresholdを変更して再評価（再学習なし）

`NEW_THRESHOLD` に値（PatchCoreの生スコア単位）を設定して実行すると、新しいRunとして保存します。

**注意:** testの結果を見ながら決めたThresholdのRunは、独立した最終評価ではありません。

In [ ]:
from anomaly_detection.experiments.pipeline import reevaluate

NEW_THRESHOLD = None  # 例: result.threshold.value * 0.9

if result and NEW_THRESHOLD is not None:
    re_result = reevaluate(result.run.run_dir, NEW_THRESHOLD)
    display(
        pd.DataFrame(
            {
                "before": result.metrics["image"],
                "after": re_result.metrics["image"],
            }
        ).loc[["recall", "precision", "f1", "accuracy", "fn", "fp"]]
    )

## 12. Experiment Result

条件（前処理・Dataset・学習条件・Threshold）を変更して再実行したRunは `03_error_analysis.ipynb` で比較できます。

In [ ]:
from anomaly_detection.experiments.run_manager import load_summary

if result:
    print("Run Directory:", result.run.run_dir)
    summary = load_summary(result.run.run_dir.parent)
    display(summary[summary["phase"] == "phase2"].tail(5))